# 03 — Feature Engineering & Timeseries Expansion
## India Tourist Crowd Forecasting System

### What this notebook does
Transforms the 12,655 static place dataset into a full time-series panel dataset
with real seasonal signals for each (place × month) combination.

**Pipeline:**
1. **Step 1 — Data Cleaning:** Fix corrupted values, drop dead columns, impute nulls
2. **Step 2 — Timeseries Expansion:** 12,515 places × 36 months = 450,540 rows
3. **Calendar Merge:** Join real festival/holiday calendar (2025-2027)
4. **Climate Merge:** Join real ERA5 per-city monthly weather
5. **Trend Parsing:** Extract real per-month Google Trends from JSON blob
6. **Interaction Features:** 5 place-type × season interactions
7. **Crowd Score Formula:** Weighted real-signal composite score
8. **Label Computation:** relative_crowd_label (Low/Medium/High per place)

**Input:** `india_crowd_enhanced_v6_FINAL.csv` (12,655 × 95)
**Output:** `india_crowd_timeseries_v2.csv` (450,540 × 106)

---
### Type of Time Series Used
This is **NOT** a classical ARIMA/LSTM model. It is a **feature-based panel data classifier**:
- Each row = one (place, month) combination
- Model learns patterns like: `heritage + festival + pleasant weather = High crowd`
- Uses real seasonal signals as features, not sequential historical data
- Correct approach since no real monthly visitor counts exist for 12,655 Indian places

**Crowd Score Formula Weights:**
```
monthly_crowd_score =
  busyness_avg            × 0.25  (real Google Popular Times)
  weather_score           × 0.20  (real ERA5 per-city weather)
  festival_boost_score    × 0.20  (real festival calendar)
  search_trend_this_month × 0.20  (real Google Trends monthly)
  review_velocity_score   × 0.10  (real review data)
  calendar_boost          × 0.05  (school vacations / holidays)
```

In [ ]:
# ── Setup ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

print('✅ Setup complete')

## Step 1 — Data Cleaning

**Bugs found and fixed in this step:**
1. **264 rows** with corrupted `num_reviews` (fractional values like 0.05 — impossible for review counts). Root cause: a normalized value was written into the raw column during an earlier merge.
2. **`search_trend_month_avg`** was a JSON blob — protected from blanket 'Unknown' fill that would corrupt JSON parsing.

In [ ]:
# ─────────────────────────────────────────────────
# STEP 1: DATA CLEANING
# ─────────────────────────────────────────────────

df = pd.read_csv('/content/drive/MyDrive/india_crowd_enhanced_v6_FINAL.csv')
print(f'BEFORE cleaning: {df.shape}')

# ── 1. Geographic validity (India bounding box) ───
INDIA_LAT = (6.0, 38.0)
INDIA_LNG = (68.0, 98.0)
missing_geo   = df['latitude'].isnull() | df['longitude'].isnull()
out_of_bounds = (~df['latitude'].between(*INDIA_LAT) |
                 ~df['longitude'].between(*INDIA_LNG)) & ~missing_geo
bad_geo = missing_geo | out_of_bounds
df = df[~bad_geo].copy()
print(f'[Geo] Dropped {bad_geo.sum()} rows | Shape: {df.shape}')

# ── 2. Rating validity ────────────────────────────
bad_rating = ~df['rating'].between(0, 5) & df['rating'].notna()
df = df[~bad_rating].copy()
print(f'[Rating] Invalid rows removed: {bad_rating.sum()}')

# ── 3. num_reviews integrity check ───────────────
# BUG FIX: 264 rows had fractional num_reviews (e.g. 0.05)
# These were corrupted by an earlier merge with normalized values
corrupted = (df['num_reviews'] < 1) | (df['num_reviews'] != df['num_reviews'].round())
print(f'[num_reviews] Corrupted rows: {corrupted.sum()}')
df.loc[corrupted, 'num_reviews'] = np.nan
df['num_reviews'] = df.groupby('place_type')['num_reviews'].transform(
    lambda x: x.fillna(x.median()))
df['num_reviews'] = df['num_reviews'].fillna(df['num_reviews'].median()).round().astype(int)
print(f'  → Fixed {corrupted.sum()} values (median per place_type)')

# ── 4. Busyness columns (0-100) ──────────────────
busy_cols = [c for c in ['busyness_avg','busyness_max','morning_avg','afternoon_avg',
             'evening_avg','night_avg','weekday_avg','weekend_avg',
             'mon_avg','tue_avg','wed_avg','thu_avg','fri_avg','sat_avg','sun_avg']
             if c in df.columns]
for c in busy_cols: df[c] = df[c].clip(0, 100)
print(f'[Busyness] Clipped {len(busy_cols)} columns to 0-100')

# ── 5. Text consistency ───────────────────────────
for c in ['place_name','city','state','zone','place_type']:
    if c in df.columns: df[c] = df[c].astype(str).str.strip()

# ── 6. Drop dead columns ─────────────────────────
DROP_COLS = ['entry_fee_inr','entry_free','dslr_allowed','significance',
    'best_time','weekly_off','best_monsoon','best_winter','best_summer',
    'best_all_year','crowd_level_final','crowd_label_final',
    'typical_open_hour','typical_close_hour','place_id_google',
    'recent_popularity_trend','entry_fee_inr_norm']
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=DROP_COLS)
print(f'[Columns] Dropped {len(DROP_COLS)} dead columns')

# ── 7. age_years → age_known flag ────────────────
if 'age_years' in df.columns:
    df['age_known'] = (df['age_years'] != 100).astype(int)
    df = df.drop(columns=['age_years'])

# ── 8. Zero-variance safety net ──────────────────
near_constant = [c for c in df.select_dtypes(include=[np.number]).columns
                 if df[c].nunique() <= 1]
if near_constant: df = df.drop(columns=near_constant)
print(f'[Variance] Zero-variance dropped: {near_constant}')

# ── 9. Null imputation ────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df.groupby('place_type')[col].transform(
            lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())

# BUG FIX: search_trend_month_avg is a JSON blob — exclude from
# blanket 'Unknown' fill which would corrupt JSON parsing later
cat_cols = df.select_dtypes(include=['object','string']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'search_trend_month_avg']
for col in cat_cols:
    if df[col].isnull().sum() > 0: df[col] = df[col].fillna('Unknown')

# ── 10. Log transforms ───────────────────────────
df['num_reviews_log'] = np.log1p(df['num_reviews'])
if 'popularity_score' in df.columns:
    df['popularity_score_log'] = np.log1p(df['popularity_score'].clip(lower=0))

# ── 11. Memory optimization ───────────────────────
mem_before = df.memory_usage(deep=True).sum()/1e6
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')
mem_after = df.memory_usage(deep=True).sum()/1e6

print(f'\n{"="*55}')
print(f'  CLEANING COMPLETE')
print(f'{"="*55}')
print(f'  Final shape    : {df.shape}')
print(f'  Nulls remaining: {df.isnull().sum().sum()} (intentional — search_trend_month_avg)')
print(f'  Memory         : {mem_before:.1f} MB → {mem_after:.1f} MB')

CLEAN_PATH = '/content/drive/MyDrive/india_crowd_cleaned_v1.csv'
df.to_csv(CLEAN_PATH, index=False)
print(f'\n💾 Saved → {CLEAN_PATH}')

## Step 2 — Timeseries Expansion + Feature Engineering

**Bugs found and fixed in this step:**
1. **Unused JSON trend data:** `search_trend_month_avg` contained real per-month Google Trends data that was never used. Fixed by parsing JSON and extracting month-specific value per row.
2. **Pandas 3.0 column drop:** `groupby().apply()` silently drops the grouping column in pandas 3.0. Fixed by using `.transform()` instead.

In [ ]:
# ─────────────────────────────────────────────────
# STEP 2: TIMESERIES EXPANSION + FEATURE ENGINEERING
# ─────────────────────────────────────────────────

df = pd.read_csv('/content/drive/MyDrive/india_crowd_cleaned_v1.csv')
calendar_lookup = pd.read_csv('/content/drive/MyDrive/calendar_lookup_2025_2027.csv')
climate_lookup  = pd.read_csv('/content/drive/MyDrive/climate_lookup_by_city_real.csv')

print(f'Base (cleaned)  : {df.shape}')
print(f'Calendar lookup : {calendar_lookup.shape}')
print(f'Climate lookup  : {climate_lookup.shape}')

# ── 0. Parse real monthly search trend ───────────
# BUG FIX: search_trend_month_avg contained a real per-month JSON blob
# e.g. {"1": 17.14, "2": 22.15, ..., "12": 57.55}
# This was sitting completely UNUSED — the crowd score only used flat annual avg
# Fix: parse JSON at place level (fast), then extract per-row after expansion
def parse_trend_json(x):
    if pd.isna(x): return [np.nan]*12
    try:
        d = json.loads(x)
        return [d.get(str(m), np.nan) for m in range(1,13)]
    except (json.JSONDecodeError, TypeError):
        return [np.nan]*12

trend_parsed = df['search_trend_month_avg'].apply(parse_trend_json)
trend_wide   = pd.DataFrame(trend_parsed.tolist(),
                             columns=[f'trend_m{m}' for m in range(1,13)],
                             index=df.index)
df = pd.concat([df, trend_wide], axis=1)
df = df.drop(columns=['search_trend_month_avg'])
parsed_ok = trend_wide.notna().any(axis=1).sum()
print(f'\n[Trends] Parsed real monthly data for {parsed_ok:,} places ({parsed_ok/len(df)*100:.1f}%)')

# ── 1. Expand: place x year x month ──────────────
YEARS  = [2025, 2026, 2027]
MONTHS = list(range(1, 13))
year_month = pd.DataFrame(
    [(y, m) for y in YEARS for m in MONTHS],
    columns=['year','month'])
df['_key']         = 1
year_month['_key'] = 1
df_exp = df.merge(year_month, on='_key').drop(columns='_key')
print(f'\n[Expand] {df.shape} → {df_exp.shape} ({len(df):,} places x {len(year_month)} year-months)')

# ── 2. Merge calendar + climate ───────────────────
df_exp = df_exp.merge(calendar_lookup, on=['year','month'], how='left')
df_exp = df_exp.merge(climate_lookup,  on=['city','month'], how='left')
print(f'[Merge] {df_exp.shape} | Nulls: calendar={df_exp["season"].isnull().sum()}, '
      f'climate={df_exp["real_avg_temp_c_month"].isnull().sum()}')

# ── 3. Cyclical month encoding ────────────────────
# WHY: Month 12 and Month 1 should be numerically CLOSE (31 days apart)
# but as raw integers they are 11 apart — sin/cos fixes this
df_exp['month_sin'] = np.sin(2 * np.pi * df_exp['month'] / 12)
df_exp['month_cos'] = np.cos(2 * np.pi * df_exp['month'] / 12)

# ── 3b. Extract this row's month-specific trend ───
# Vectorized numpy indexing — fast even on 450K+ rows
trend_cols   = [f'trend_m{m}' for m in range(1,13)]
trend_matrix = df_exp[trend_cols].values
month_idx    = (df_exp['month'].values - 1).astype(int)
df_exp['search_trend_this_month'] = trend_matrix[np.arange(len(df_exp)), month_idx]
df_exp = df_exp.drop(columns=trend_cols)

# Fallback: this-month → annual avg → global median
df_exp['search_trend_this_month'] = df_exp['search_trend_this_month'].fillna(
    df_exp['search_trend_score'])
df_exp['search_trend_this_month'] = df_exp['search_trend_this_month'].fillna(
    df_exp['search_trend_this_month'].median())
print(f'[Trends] search_trend_this_month extracted (real for ~{df_exp["trend_data_available"].mean()*100:.0f}% of rows)')

# ── 4. Interaction features ───────────────────────
# WHY: Different place types respond differently to seasons
# Beaches peak in WINTER (not festivals)
# Heritage sites peak during FESTIVALS
# Museums are indoor refuge during MONSOON
df_exp['heritage_x_long_weekend']    = df_exp['is_heritage']    * df_exp['is_long_weekend_month']
df_exp['religious_x_festival']       = df_exp['is_religious']   * df_exp['has_major_festival']
df_exp['beach_x_monsoon']            = df_exp['is_beach']       * df_exp['real_is_monsoon_month'].fillna(0)
df_exp['hillstation_x_summer']       = df_exp.get('is_hill_station', pd.Series(0, index=df_exp.index)) * \
                                        (df_exp['season'] == 'Summer').astype(int)
df_exp['heritage_x_school_vacation'] = df_exp['is_heritage']    * df_exp['is_school_vacation']
print(f'[Interactions] 5 interaction features added')

# ── 5. Weather suitability score ──────────────────
# Place-type × real weather category preference matrix
PLACE_TYPE_WEATHER_PREF = {
    'Beach'         : {'Pleasant':1.3,'Hot':0.9,'Hot/Rainy':0.5,'Rainy':0.3,'Cold':0.7},
    'Hill Station'  : {'Pleasant':1.2,'Hot':1.3,'Hot/Rainy':0.8,'Rainy':0.6,'Cold':0.7},
    'Wildlife'      : {'Pleasant':1.3,'Hot':0.8,'Hot/Rainy':0.6,'Rainy':0.4,'Cold':1.1},
    'Heritage'      : {'Pleasant':1.3,'Hot':0.8,'Hot/Rainy':0.6,'Rainy':0.5,'Cold':1.1},
    'Religious'     : {'Pleasant':1.1,'Hot':1.0,'Hot/Rainy':0.9,'Rainy':0.8,'Cold':1.0},
    'Museum'        : {'Pleasant':1.0,'Hot':1.1,'Hot/Rainy':1.0,'Rainy':1.1,'Cold':0.9},
    'Market'        : {'Pleasant':1.1,'Hot':0.9,'Hot/Rainy':0.8,'Rainy':0.7,'Cold':1.0},
    'Park'          : {'Pleasant':1.3,'Hot':0.8,'Hot/Rainy':0.5,'Rainy':0.4,'Cold':0.9},
    'Nature'        : {'Pleasant':1.3,'Hot':0.8,'Hot/Rainy':0.6,'Rainy':0.5,'Cold':0.9},
    'Viewpoint'     : {'Pleasant':1.3,'Hot':0.9,'Hot/Rainy':0.5,'Rainy':0.4,'Cold':0.8},
    'Cave'          : {'Pleasant':1.0,'Hot':1.1,'Hot/Rainy':1.0,'Rainy':1.0,'Cold':0.9},
    'Amusement Park': {'Pleasant':1.3,'Hot':0.9,'Hot/Rainy':0.6,'Rainy':0.5,'Cold':0.8},
    'Tourist Spot'  : {'Pleasant':1.2,'Hot':0.9,'Hot/Rainy':0.7,'Rainy':0.6,'Cold':0.9},
}
DEFAULT_PREF = PLACE_TYPE_WEATHER_PREF['Tourist Spot']

def weather_mult(row):
    pref = PLACE_TYPE_WEATHER_PREF.get(row['place_type'], DEFAULT_PREF)
    cat  = row['real_weather_category'] if pd.notna(row.get('real_weather_category')) else 'Pleasant'
    return pref.get(cat, 1.0)

df_exp['weather_score'] = (df_exp.apply(weather_mult, axis=1) * 50).clip(0, 100)
print(f'[Weather] weather_score computed from REAL ERA5 per-city weather')

# ── 6. Festival + calendar boost ─────────────────
df_exp['festival_boost_score'] = (
    df_exp['num_festivals_in_month'].fillna(0) * 10 +
    df_exp['has_major_festival'].fillna(0) * 20
).clip(0, 50)
df_exp['calendar_boost'] = (
    df_exp['is_school_vacation'].fillna(0) * 10 +
    df_exp['is_long_weekend_month'].fillna(0) * 5
)

# ── 7. Final crowd score (uses REAL monthly trend) ─
# Monthly crowd score formula with weighted real signals
df_exp['busyness_avg']  = df_exp['busyness_avg'].fillna(30)
trend_norm              = df_exp['search_trend_this_month'].clip(0, 100)
df_exp['monthly_crowd_score'] = (
    df_exp['busyness_avg']          * 0.25 +
    df_exp['weather_score']         * 0.20 +
    df_exp['festival_boost_score']  * 0.20 +
    trend_norm                      * 0.20 +
    df_exp['review_velocity_score'] * 0.10 +
    df_exp['calendar_boost']        * 0.05
).round(2)
print(f'[Score] monthly_crowd_score uses REAL monthly search trend (20% weight)')

# ── 8. Labels — ROBUST transform-based ───────────
# BUG FIX: pandas 3.0 groupby().apply() silently drops grouping column
# Fix: use .transform() which broadcasts back to original index
def compute_group_labels(data, group_col, score_col, seed_offset=0):
    np.random.seed(42 + seed_offset)
    scores = data[score_col]
    p33 = scores.groupby(data[group_col]).transform(lambda x: x.quantile(0.33))
    p66 = scores.groupby(data[group_col]).transform(lambda x: x.quantile(0.66))
    flat = (p33 == p66)
    if flat.any():
        noisy  = scores + np.random.normal(0, 0.5, len(data))
        p33_n  = noisy.groupby(data[group_col]).transform(lambda x: x.quantile(0.33))
        p66_n  = noisy.groupby(data[group_col]).transform(lambda x: x.quantile(0.66))
        scores = scores.where(~flat, noisy)
        p33    = p33.where(~flat, p33_n)
        p66    = p66.where(~flat, p66_n)
    return np.select([scores <= p33, scores <= p66], ['Low','Medium'], default='High')

df_exp['monthly_crowd_label']  = compute_group_labels(df_exp, 'place_type', 'monthly_crowd_score', 0)
df_exp['relative_crowd_label'] = compute_group_labels(df_exp, 'place_id',   'monthly_crowd_score', 1)
print(f'[Labels] monthly_crowd_label + relative_crowd_label computed')
print(f'         place_id preserved: {"place_id" in df_exp.columns} ✅')

# ── Summary ───────────────────────────────────────
print(f'\n{"="*55}')
print(f'  FEATURE ENGINEERING COMPLETE')
print(f'{"="*55}')
print(f'  Final shape : {df_exp.shape}')
print(f'  place_id    : present={"place_id" in df_exp.columns}, '
      f'nulls={df_exp["place_id"].isnull().sum()}, '
      f'unique={df_exp["place_id"].nunique():,}')
print(f'\n  monthly_crowd_label distribution:')
print(df_exp['monthly_crowd_label'].value_counts(normalize=True).round(3))
print(f'\n  relative_crowd_label distribution:')
print(df_exp['relative_crowd_label'].value_counts(normalize=True).round(3))

TS_PATH = '/content/drive/MyDrive/india_crowd_timeseries_v2.csv'
df_exp.to_csv(TS_PATH, index=False)
print(f'\n💾 Saved → {TS_PATH}')
print(f'   Memory: {df_exp.memory_usage(deep=True).sum()/1e6:.1f} MB')

## Verification — Spot Check
Verify that seasonal patterns make sense for well-known places.

In [ ]:
# ── Spot check: Taj Mahal seasonal pattern ────────
print('Spot check — Taj Mahal 2026:')
taj = df_exp[
    (df_exp['place_name'] == 'Taj Mahal') &
    (df_exp['year'] == 2026)
].sort_values('month')

if len(taj) > 0:
    print(taj[['month','season','real_weather_category',
               'has_major_festival','monthly_crowd_score',
               'relative_crowd_label']].to_string(index=False))
    print('\nExpected: Oct/Nov = High (Dussehra season + pleasant weather)')
    print('          Jun/Jul = Low  (monsoon season)')
else:
    print('Taj Mahal not found in dataset (check place name spelling)')

# ── Dataset shape verification ────────────────────
print(f'\n✅ Verification:')
print(f'   Rows    : {len(df_exp):,} (expect 450,540)')
print(f'   Cols    : {len(df_exp.columns)} (expect ~106)')
print(f'   Nulls   : {df_exp.isnull().sum().sum()}')
print(f'   Places  : {df_exp["place_id"].nunique():,} (expect 12,515)')
print(f'\n✅ Proceed to 04_preprocessing.ipynb!')